# 05 — Final Model on Full Data

Takes whichever model/config scored highest F1 in `04_evaluate_compare.ipynb` (read from `data/model_comparison_results.csv`, produced after `03b_hyperparameter_tuning.ipynb`'s tuned models — and the XGBoost/LightGBM candidates — have been folded into that comparison) and retrains it on the **full 253,680-row raw dataset** instead of the 40,000-row stratified sample used everywhere else in the pipeline, to check whether more data improves generalization.

Requires `04_evaluate_compare.ipynb` to have been run (so `model_comparison_results.csv` reflects the full model lineup, tuned + untuned + gradient boosting) and its winning model's `.pkl` to exist under `data/models/`.

In [ ]:
import json
import pickle

import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from config import (
    DATA_DIR, RAW_CSV, DROPPED_FEATURES_JSON, TARGET_COL, TEST_SIZE, RANDOM_STATE,
)

MODELS_DIR = DATA_DIR / "models"
BASE_NUMERIC_COLS = ["BMI", "MentHlth", "PhysHlth"]
OPTIONAL_ENGINEERED_NUMERIC_COLS = ["RiskScore", "TotalUnhealthyDays"]

## Identify the winning model

Reads `data/model_comparison_results.csv` (produced by `04_evaluate_compare.ipynb`) and picks the row with the highest F1. The model label encodes both the model family and its training-set suffix (`orig`, `smote`, `orig_tuned`, `smote_tuned`), which we parse back into the `data/models/{name}_{suffix}.pkl` filename convention used throughout `03_train_models.ipynb` / `03b_hyperparameter_tuning.ipynb`.

In [ ]:
results_path = DATA_DIR / "model_comparison_results.csv"
if not results_path.exists():
    raise FileNotFoundError(
        f"Could not find {results_path}. Run 04_evaluate_compare.ipynb first."
    )

results_df = pd.read_csv(results_path).sort_values("F1", ascending=False)
winner_label = results_df.iloc[0]["Model"]
print(f"Winning model (highest F1 on 40k-sample test set): {winner_label}")
results_df.head(5)

In [ ]:
# Map a display label like "Random Forest (SMOTE, tuned)" back to the
# (model_key, suffix) used for data/models/{model_key}_{suffix}.pkl.
LABEL_TO_KEY = {
    "Decision Tree": "decision_tree",
    "Naive Bayes": "naive_bayes",
    "k-NN": "knn",
    "Random Forest": "random_forest",
    "AdaBoost": "adaboost",
    "XGBoost": "xgboost",
    "LightGBM": "lightgbm",
}


def parse_label(label):
    name_part, paren = label.split(" (")
    paren = paren.rstrip(")")
    tokens = [t.strip() for t in paren.split(",")]
    balance = "smote" if "SMOTE" in tokens[0] else "orig"
    tuned = "tuned" in tokens
    suffix = f"{balance}_tuned" if tuned else balance
    model_key = LABEL_TO_KEY[name_part]
    return model_key, suffix, balance


winner_key, winner_suffix, winner_balance = parse_label(winner_label)
winner_path = MODELS_DIR / f"{winner_key}_{winner_suffix}.pkl"
print(f"Loading winning model from: {winner_path}")

with open(winner_path, "rb") as f:
    winner_model_sample = pickle.load(f)

winner_params = winner_model_sample.get_params()
print(f"Model class: {type(winner_model_sample).__name__}")
print(f"Params: {winner_params}")

## Load full raw dataset and rebuild the same feature set

Uses `config.RAW_CSV` directly (253,680 rows) rather than the 40k stratified sample. Reapplies the exact feature engineering `01b_feature_engineering.ipynb` decided on (`BMI_Category` WHO-cutoff bins, `RiskScore` composite, and the raw-column drops recorded in `data/dropped_features.json`), so the feature schema matches what the winning model was trained on.

In [ ]:
if not RAW_CSV.exists():
    raise FileNotFoundError(f"Could not find {RAW_CSV}. Run 00_data_load.ipynb first.")

df = pd.read_csv(RAW_CSV)
print(f"Full raw dataset shape: {df.shape}")

with open(DROPPED_FEATURES_JSON) as f:
    decision_record = json.load(f)

engineered_kept = decision_record["engineered_features_kept"]
raw_dropped = decision_record["raw_features_dropped"]
print(f"Engineered features kept (from 01b): {engineered_kept}")
print(f"Raw features dropped (from 01b): {raw_dropped}")

full_df = df.drop(columns=raw_dropped).copy()

if "BMI_Category" in engineered_kept:
    bmi_bins = [0, 18.5, 25, 30, 40, df["BMI"].max() + 1]
    bmi_labels = ["Underweight", "Normal", "Slightly_Overweight", "Overweight", "Obese"]
    bmi_category = pd.cut(df["BMI"], bins=bmi_bins, labels=bmi_labels)
    full_df = pd.concat([full_df, pd.get_dummies(bmi_category, prefix="BMI")], axis=1)

if "RiskScore" in engineered_kept:
    risk_flags = ["HighBP", "HighChol", "Stroke", "HeartDiseaseorAttack", "DiffWalk", "Smoker"]
    full_df["RiskScore"] = df[risk_flags].sum(axis=1)

if "TotalUnhealthyDays" in engineered_kept:
    full_df["TotalUnhealthyDays"] = df["MentHlth"] + df["PhysHlth"]

if "IsElderly" in engineered_kept:
    full_df["IsElderly"] = (df["Age"] >= 10).astype(int)

print(f"Full engineered dataset shape: {full_df.shape}")
full_df.head()

## Split, scale, resample (same conventions as `02_preprocessing.ipynb`)

Same `test_size`/`stratify`/`random_state` convention from `config.py`, `StandardScaler` fit on train only, and — since the comparison should isolate the effect of data volume rather than also changing the imbalance strategy — the same resampling choice (SMOTE or none) the winning 40k-sample config used.

In [ ]:
NUMERIC_COLS = [c for c in BASE_NUMERIC_COLS + OPTIONAL_ENGINEERED_NUMERIC_COLS if c in full_df.columns]
print(f"Numeric columns to scale: {NUMERIC_COLS}")

X = full_df.drop(columns=[TARGET_COL])
y = full_df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train = X_train.copy()
X_test = X_test.copy()
X_train[NUMERIC_COLS] = scaler.fit_transform(X_train[NUMERIC_COLS])
X_test[NUMERIC_COLS] = scaler.transform(X_test[NUMERIC_COLS])

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print("Test class balance:")
print(y_test.value_counts(normalize=True))

if winner_balance == "smote":
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train, y_train = smote.fit_resample(X_train, y_train)
    print(f"SMOTE-balanced train shape: {X_train.shape}")
else:
    print("Winning config trained on the original (non-resampled) training set — no resampling applied here either.")

## Train the winning model on the full-data split

Reuses the exact hyperparameters read off the winning pickled model (whether tuned by Optuna in `03b_hyperparameter_tuning.ipynb` or one of the fixed baseline configs from `03_train_models.ipynb`) — this notebook retrains with those params on more data, it does not re-tune.

In [ ]:
final_model = type(winner_model_sample)(**winner_params)
final_model.fit(X_train, y_train)

print(f"Trained {type(final_model).__name__} on {X_train.shape[0]} rows.")

## Evaluate on the full-data held-out test set

In [ ]:
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

full_data_metrics = {
    "Model": f"{winner_label} — full data",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, y_proba),
}

print("Full-data (253,680 rows) result:")
for k, v in full_data_metrics.items():
    print(f"  {k}: {v}")

## Compare against the same model on the 40k-row sample

In [ ]:
sample_row = results_df[results_df["Model"] == winner_label].iloc[0]

comparison = pd.DataFrame([
    {"Dataset": "40k stratified sample", **sample_row[["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]].to_dict()},
    {"Dataset": "Full 253,680-row dataset", **{k: v for k, v in full_data_metrics.items() if k != "Model"}},
])
comparison

## Save the final model

In [ ]:
final_path = MODELS_DIR / "final_model_full_data.pkl"
with open(final_path, "wb") as f:
    pickle.dump(final_model, f)

comparison.to_csv(DATA_DIR / "final_model_full_data_comparison.csv", index=False)

print(f"Saved final model to {final_path}")
print(f"Saved comparison table to {DATA_DIR / 'final_model_full_data_comparison.csv'}")